In [28]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx

subs_df = pd.read_csv("data/B28002 Presence and Types of Internet Subscriptions in Household/ACSDT5Y2024.B28002-Data.csv", header=1)
income_df = pd.read_csv("data/B28004 Household Income in the Last 12 Months (in 2024 Inflation-Adjusted Dollars) by Presence and Type of Internet Subscription in Household/ACSDT5Y2024.B28004-Data.csv", header=1)
age_df = pd.read_csv("data/B28005 Age by Presence of a Computer and Types of Internet Subscription in Household/ACSDT5Y2024.B28005-Data.csv", header=1)
labor_df = pd.read_csv("data/B28007 Labor Force Status by Presence of a Computer and Types of Internet Subscription in Household/ACSDT5Y2024.B28007-Data.csv", header=1)

In [29]:
subs_df["GEOID"] = subs_df["Geography"].str[9:]
subs_df = subs_df.drop(columns=["Geography", "Geographic Area Name"])
subs_df.insert(0,"GEOID", subs_df.pop("GEOID"))

status = {
    "With an Internet subscription": "sub",
    "Internet access without a subscription": "access_no_sub",
    "No Internet access": "no_access" 
}
sub_type = {
    "Dial-up with no other type of Internet subscription": "dialup_only",
    "Broadband of any type": "broadband_any",
    "Cellular data plan": "cell",
    "Cellular data plan with no other type of Internet subscription": "only",
    "Broadband such as cable, fiber optic or DSL": "wired",
    "Broadband such as cable, fiber optic or DSL with no other type of Internet subscription": "only",
    "Satellite Internet service": "sat",
    "Satellite Internet service with no other type of Internet subscription": "only",
    "Other service with no other type of Internet subscription": "other_only",
}

columns = subs_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        continue
    if "Unnamed" in label:
        subs_df = subs_df.drop(columns=subs_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [status.get(p, sub_type.get(p)) for p in parts[2:]]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    subs_df = subs_df.rename(columns={subs_df.columns[i]: name})

In [30]:
income_df["GEOID"] = income_df["Geography"].str[9:]
income_df = income_df.drop(columns=["Geography", "Geographic Area Name"])
income_df.insert(0,"GEOID", income_df.pop("GEOID"))

brackets = {
    "Less than $10,000:": "lt10k",
    "$10,000 to $19,999:": "10to20k",
    "$20,000 to $34,999:": "20to35k",
    "$35,000 to $49,999:": "35to50k",
    "$50,000 to $74,999:": "50to75k",
    "$75,000 or more:": "75kplus",
}
sub_type = {
    "With dial-up Internet subscription alone": "dialup",
    "With a broadband Internet subscription": "broadband",
    "Without an Internet subscription": "noSub",
}

columns = income_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        continue
    if "Unnamed" in label:
        income_df = income_df.drop(columns=income_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [brackets.get(p, sub_type.get(p)) for p in parts[2:]]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    income_df = income_df.rename(columns={income_df.columns[i]: name})

In [31]:
age_df["GEOID"] = age_df["Geography"].str[9:]
age_df = age_df.drop(columns=["Geography", "Geographic Area Name"])
age_df.insert(0,"GEOID", age_df.pop("GEOID"))

ages = {
    "Under 18 years:": "u18",
    "18 to 64 years:": "18_64",
    "65 years and over:": "65_plus", 
}
computer_sub = {
    "Has a computer:": "computer",
    "No computer": "no_computer",
    "With dial-up Internet subscription alone": "dialup",
    "With a broadband Internet subscription": "broadband",
    "Without an Internet subscription": "no_sub",
}

columns = age_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        continue
    if "Unnamed" in label:
        age_df = age_df.drop(columns=age_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [status.get(p, computer_sub.get(p)) for p in parts[2:]]
    pieces = [p for p in pieces if p is not None]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    age_df = age_df.rename(columns={age_df.columns[i]: name})

In [32]:
labor_df["GEOID"] = labor_df["Geography"].str[9:]
labor_df = labor_df.drop(columns=["Geography", "Geographic Area Name"])
labor_df.insert(0,"GEOID", labor_df.pop("GEOID"))

labor = {
    "In the civilian labor force:": "lf",
    "Employed:": "employed",
    "Unemployed:": "unemployed",
    "Not in labor force:": "not_lf", 
}
computer_sub = {
    "Has a computer:": "computer",
    "No computer": "no_computer",
    "With dial-up Internet subscription alone": "dialup",
    "With a broadband Internet subscription": "broadband",
    "Without an Internet subscription": "no_sub",
}

columns = labor_df.columns.tolist()

for i, label in enumerate(columns):
    if label == "GEOID":
        continue
    if "Unnamed" in label:
        labor_df = labor_df.drop(columns=labor_df.columns[i])
        continue 
    parts = label.split("!!")
    pieces = [status.get(p, computer_sub.get(p)) for p in parts[2:]]
    pieces = [p for p in pieces if p is not None]
    name = "_".join(pieces) if pieces else "total"
    if parts[0] == "Margin of Error":
        name += "_moe"
    labor_df = labor_df.rename(columns={labor_df.columns[i]: name})

In [33]:
subs_df.head()

,GEOID,total,total_moe,sub,sub_moe,sub_dialup_only,sub_dialup_only_moe,sub_broadband_any,sub_broadband_any_moe,sub_cell,...,sub_sat,sub_sat_moe,sub_sat_only,sub_sat_only_moe,sub_other_only,sub_other_only_moe,access_no_sub,access_no_sub_moe,no_access,no_access_moe
0,37,4282345,9120,3862460,12406,5056,457,3857404,12386,3601349,...,262566,4373,16715,1083,4427,681,100242,3374,319643,6043
1,37001020100,2052,211,1760,202,0,14,1760,202,1670,...,43,38,0,14,0,14,78,64,214,124
2,37001020200,1639,291,1457,281,0,14,1457,281,1100,...,19,32,0,14,0,14,15,18,167,99
3,37001020301,1675,272,1406,274,0,14,1406,274,1385,...,100,85,0,14,0,14,0,14,269,151
4,37001020302,1190,177,987,205,0,14,987,205,977,...,178,105,0,14,0,14,23,33,180,117


In [34]:
income_df.head()

,GEOID,total,total_moe,lt10k,lt10k_moe,lt10k_dialup,lt10k_dialup_moe,lt10k_broadband,lt10k_broadband_moe,lt10k_noSub,...,50to75k_noSub,50to75k_noSub_moe,75kplus,75kplus_moe,75kplus_dialup,75kplus_dialup_moe,75kplus_broadband,75kplus_broadband_moe,75kplus_noSub,75kplus_noSub_moe
0,37,4282345,9120,214424,3789,343,118,164746,3625,49335,...,56493,2132,2070458,11495,1246,255,1995033,11707,74179,2739
1,37001020100,2052,211,199,126,0,14,133,111,66,...,38,46,707,197,0,14,646,182,61,77
2,37001020200,1639,291,342,268,0,14,321,269,21,...,0,14,162,100,0,14,143,97,19,31
3,37001020301,1675,272,120,73,0,14,89,68,31,...,21,35,504,199,0,14,493,193,11,24
4,37001020302,1190,177,90,76,0,14,84,75,6,...,49,50,352,134,0,14,295,130,57,70


In [35]:
age_df.head()

,GEOID,total,total_moe,total,total_moe,computer,computer_moe,computer_dialup,computer_dialup_moe,computer_broadband,...,computer,computer_moe,computer_dialup,computer_dialup_moe,computer_broadband,computer_broadband_moe,computer_no_sub,computer_no_sub_moe,no_computer,no_computer_moe
0,37,10451237,*****,2331363,555,2304151,2617,676,260,2223734,...,1612209,4389,4990,604,1486464,5116,120755,3001,186138,4119
1,37001020100,4164,488,722,149,722,149,0,14,722,...,631,132,0,14,583,129,48,55,110,69
2,37001020200,4123,823,1083,448,1083,448,0,14,1083,...,442,122,0,14,375,121,67,36,115,101
3,37001020301,3984,708,1199,438,1199,438,0,14,1199,...,553,141,0,14,436,129,117,87,50,47
4,37001020302,3350,511,867,279,856,276,0,14,801,...,452,165,0,14,437,164,15,24,166,118


In [36]:
labor_df.head()

,GEOID,total,total_moe,total,total_moe,total,total_moe,computer,computer_moe,computer_dialup,...,computer,computer_moe,computer_dialup,computer_dialup_moe,computer_broadband,computer_broadband_moe,computer_no_sub,computer_no_sub_moe,no_computer,no_computer_moe
0,37,8324255,3485,5294232,12528,5048341,12886,4974558,12878,2346,...,2804607,12937,5499,648,2604893,11997,194215,4947,225416,5841
1,37001020100,3457,462,2176,378,2085,358,2019,319,0,...,1150,287,0,14,1081,281,69,65,131,77
2,37001020200,3118,547,1932,347,1605,399,1568,408,0,...,1097,326,0,14,1029,328,68,49,89,68
3,37001020301,2994,522,1873,389,1695,360,1682,357,0,...,1044,284,0,14,927,265,117,87,77,64
4,37001020302,2523,380,1617,268,1487,254,1355,284,0,...,770,222,0,14,740,224,30,38,136,107
